In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings,ChatOllama
from dotenv import load_dotenv
import os
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
load_dotenv()

e:\LangGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
def load_pdf_files_from_directory(directory_path):
    loader = DirectoryLoader(directory_path, glob="*.pdf", loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [3]:
DATA_PATH = r"E:/LangGraph/17. Project/books"
OLLAMA_MODEL="deepseek-v3.1:671b-cloud"
OLLAMA_EMBEDDING_MODEL="nomic-embed-text"

In [4]:
documents=load_pdf_files_from_directory(DATA_PATH)
print(f"Number of documents loaded: {len(documents)}")

Number of documents loaded: 66


In [5]:
# ------------
# splitting documents into chunks
# ------------
def split_docs(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    return chunks


In [6]:
# ------------
# creating embedder
# ------------
def get_embeddings():
    embeddings = OllamaEmbeddings(
    model=OLLAMA_EMBEDDING_MODEL,
    )
    return embeddings


In [7]:
from pinecone import Pinecone 

MyPineCone= Pinecone(api_key=os.getenv("PINECONE_API_KEY").strip())

In [8]:
MyPineCone.list_indexes()

[
    {
        "name": "langgraph-chatbot",
        "metric": "cosine",
        "host": "langgraph-chatbot-70j87ew.svc.aped-4627-b74a.pinecone.io",
        "spec": {
            "serverless": {
                "region": "us-east-1",
                "cloud": "aws",
                "read_capacity": {
                    "mode": "OnDemand",
                    "status": {
                        "state": "Ready",
                        "current_shards": null,
                        "current_replicas": null
                    }
                }
            }
        },
        "status": {
            "ready": true,
            "state": "Ready"
        },
        "vector_type": "dense",
        "dimension": 768,
        "deletion_protection": "disabled",
        "tags": null
    }
]

In [9]:
from pinecone import Pinecone, ServerlessSpec


INDEX_NAME = "langgraph-chatbot"

def get_pinecone_index(pc,index_name):
    existing_indexes = [idx.name for idx in pc.list_indexes()]
    if index_name not in existing_indexes:
        print("⚙️ Creating Pinecone index...")
        pc.create_index(
            name=index_name,
            dimension=768,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        print(f"✅ Index '{index_name}' created")
    return pc.Index(index_name)

In [10]:
from langchain_pinecone import PineconeVectorStore

def get_vectorstore(pc, embedding, index_name):
    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name not in existing_indexes:
        print("Pinecone index does not exist. Creating a new index...")
        get_pinecone_index(pc, index_name)

    return PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )

In [11]:
def add_documents_to_vectorstore(vectorstore,documents):
    vectorstore.add_documents(documents)
    print("Documents added to Pinecone vector store.")

In [12]:
def delete_all_documents(pc, index_name):
    index = pc.Index(index_name)
    index.delete(delete_all=True)
    print(f"🗑️ All documents deleted from index '{index_name}'")

In [13]:
def delete_index(pc, index_name):
    pc.delete_index(index_name)
    print(f"🔥 Index '{index_name}' fully deleted")

In [14]:
# delete_all_documents(MyPineCone, INDEX_NAME)

In [15]:
def reset_index(pc, index_name, dimension=768):
    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name in existing_indexes:
        pc.delete_index(index_name)
        print("🗑️ Old index deleted")

    get_pinecone_index(pc, index_name)
    print("✅ Fresh index created")

In [16]:
# create vector store
chunks=split_docs(load_pdf_files_from_directory(DATA_PATH))

In [17]:
print(len(chunks))

66


In [18]:
vectorstore =get_vectorstore(MyPineCone,get_embeddings(),INDEX_NAME)

In [19]:
add_documents_to_vectorstore(vectorstore,chunks)


Documents added to Pinecone vector store.


In [20]:
def get_documents_from_vectorstore(vectorstore, query, k=4):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    docs = retriever.get_relevant_documents(query)
    return docs

In [21]:
def get_llm():
    llm = ChatOllama(
        model=OLLAMA_MODEL,
        temperature=0
    )
    return llm

In [22]:
PROMPT = PromptTemplate(
    template="""
You are an AI assistant. Use the following context to answer the question.
If you don't know the answer, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

In [23]:
def get_qa_chain(vectorstore):
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=get_llm(),
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": PROMPT}
    )
    return qa_chain

In [24]:
qa_chain = get_qa_chain(vectorstore)

In [25]:


while True:
    query = input("\nAsk a question (or type 'exit'): ")
    if query.lower() == "exit":
        break

    result = qa_chain.invoke({"query": query})

    print("\nAnswer:\n", result["result"])
    
    print("\nSources:")
    for doc in result["source_documents"]:
        print("-", doc.metadata.get("source", "Unknown"))


Answer:
 Based on the context provided, Natural Language Processing (NLP) is a field of computer science and artificial intelligence focused on the interactions between computers and human (natural) languages. Its primary concern is programming computers to effectively process large amounts of natural language data.

The context further explains that NLP involves developing computational models of human language processing and is an interdisciplinary field also known by other names such as Speech and Language Processing, human language technology, computational linguistics, and speech recognition and synthesis.

Sources:
- E:\LangGraph\17. Project\books\Introduction to NLP.pdf
- E:\LangGraph\17. Project\books\Introduction to NLP.pdf
- E:\LangGraph\17. Project\books\Word Vectors-I.pdf
- E:\LangGraph\17. Project\books\Introduction to NLP.pdf

Answer:
 I don't know the answer to that question based on the context provided. The context only contains information about Word2Vec and word vec